# spinelab — анализ МРТ позвоночника в Colab

**Это исследовательский инструмент, а не диагноз.** Всё, что он выдаёт, требует проверки
врачом-рентгенологом на исходных изображениях. Отдельные этапы — эвристики по яркости
пикселей; в отчёте они помечены отдельно и не являются находками.

## Что делает
DICOM → NIfTI → сегментация (SPINEPS, TotalSpineSeg, TotalSegmentator MRI) →
совмещение серий → измерения (геометрия, мышцы, канал, диски, фасеточные зоны) →
один HTML-отчёт с уровнями доказательности.

На A100 (Colab Pro) имеет смысл профиль **QUALITY** в ячейке 3: он не меняет
определения измерений, а тратит GPU-время на то, чтобы знать их надёжность —
вторая независимая модель позвонков даёт согласие по уровням, а зеркальная TTA
проверяет, устойчиво ли модель определяет лево/право. Последнее прямо относится к
вопросу «слева или справа»: если определение стороны неустойчиво, все сравнения
сторон в этом прогоне недействительны, и отчёт это скажет.

## Порядок работы (4 ячейки, слабый интернет учтён)
1. **Подключить Drive и кэш** — веса моделей (~6–8 ГБ) скачиваются один раз и живут в Drive.
   После обрыва соединения повторный запуск ничего не докачивает.
2. **Установить окружение** — одна идемпотентная ячейка. Ядро не перезапускается принудительно.
3. **Запустить пайплайн** — по этапам, с продолжением с места обрыва (`--force` для пересчёта).
4. **Посмотреть отчёт** — HTML открывается здесь же и копируется в Drive.

## Данные пациента
Положите ZIP с DICOM **в Drive**, а не в публичный git-репозиторий: в заголовках DICOM
есть ФИО, дата рождения и учреждение. Ячейка 3 печатает, какие идентифицирующие теги
нашлись; `python -m spinelab deid` делает деидентифицированную копию.

> Runtime → Change runtime type → **GPU** (T4 хватает; A100/H100 быстрее).

In [ ]:
# @title 1 · Drive, кэш весов и исходники {display-mode:"form"}
DRIVE_ROOT = "/content/drive/MyDrive"  # @param {type:"string"}
CACHE_DIR  = "/content/drive/MyDrive/spinelab-cache"  # @param {type:"string"}
BRANCH     = "qa/2026-07-refactor"  # @param {type:"string"}
MOUNT_DRIVE = True  # @param {type:"boolean"}

import os, subprocess, sys
from pathlib import Path

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

REPO_URL = "https://github.com/omarnuri/MRI-reseqrch.git"
REPO_DIR = Path("/content/spinelab-src")

# Sparse, blobless clone: the repository still contains a ~35 MB DICOM archive and
# a 1.7 MB notebook with embedded outputs, and neither is needed to run anything.
if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                    "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "sparse-checkout", "init", "--no-cone"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "sparse-checkout", "set",
                    "/spinelab/*", "/tests/*", "/docs/*", "/pyproject.toml", "/README.md"],
                   check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{BRANCH}"], check=True)

sys.path.insert(0, str(REPO_DIR))
Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)

# Drop any already-imported spinelab modules: updating files on disk does nothing
# for a kernel that has the old ones cached, so a "pulled fix" would not take effect.
for _name in [n for n in sys.modules if n == "spinelab" or n.startswith("spinelab.")]:
    del sys.modules[_name]

sha = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
                     capture_output=True, text=True).stdout.strip()
print(f"spinelab source: {REPO_DIR} @ {sha}")
print(f"weights cache:   {CACHE_DIR}")
for name in ("spineps", "totalsegmentator", "totalspineseg", "huggingface"):
    d = Path(CACHE_DIR) / "weights" / name
    size_gb = sum(f.stat().st_size for f in d.rglob("*") if f.is_file()) / 1e9 if d.exists() else 0.0
    print(f"  cached {name:18s} {size_gb:5.2f} GB")

In [ ]:
# @title 2 · Окружение (идемпотентно, без перезапуска ядра) {display-mode:"form"}
INSTALL_SEGMENTATION = True  # @param {type:"boolean"}
FORCE_REINSTALL = False  # @param {type:"boolean"}

import importlib, subprocess, sys
from pathlib import Path

def sh(*args):
    p = subprocess.run(list(args), capture_output=True, text=True)
    if p.returncode != 0:
        # Show enough of the failure to diagnose it. Printing only the last line
        # hides pip's actual resolution error, which is what matters.
        tail = (p.stderr or p.stdout).strip().splitlines()[-12:]
        print("   ! command failed:", " ".join(args)[:120])
        for line in tail:
            print("     ", line[:160])
    return p.returncode == 0

def pip(*pkgs, upgrade=False):
    args = [sys.executable, "-m", "pip", "install", "-q"]
    if upgrade:
        args.append("--upgrade")
    ok = sh(*args, *pkgs)
    print(f"    {'ok' if ok else 'FAILED'}: {' '.join(pkgs)}")
    return ok

CORE = ["nibabel", "pydicom", "SimpleITK", "pandas"]
# SimpleITK is what the `register` stage uses to align the coronal fat-suppressed
# series with the sagittal T2 the masks come from.
SEGMENTATION = ["nnunetv2>=2.8.1", "SPINEPS>=2.0.0", "totalspineseg>=20260623",
                "TotalSegmentator>=2.16.0"]
REQUIRED = ["nibabel", "pydicom", "SimpleITK", "pandas"]
if INSTALL_SEGMENTATION:
    REQUIRED += ["nnunetv2", "spineps", "totalspineseg", "totalsegmentator"]

def importable(module):
    for name in list(sys.modules):
        if name == module or name.startswith(module + "."):
            del sys.modules[name]
    try:
        importlib.invalidate_caches()
        m = importlib.import_module(module)
        return True, getattr(m, "__version__", "present")
    except Exception as exc:
        return False, str(exc)[:110]

print("apt: dcm2niix, unzip")
sh("apt-get", "-qq", "update")
sh("apt-get", "-qq", "install", "-y", "dcm2niix", "unzip")

# State is checked, never assumed. A marker file written after a FAILED install once
# made this cell report "already installed" for a whole session while every tool was
# missing — and the pipeline then skipped every stage.
missing = [m for m in REQUIRED if not importable(m)[0]]
if missing and not FORCE_REINSTALL:
    print(f"\nmissing: {', '.join(missing)} — installing")
if FORCE_REINSTALL:
    print("\nFORCE_REINSTALL — reinstalling everything")

if missing or FORCE_REINSTALL:
    print("python: core I/O")
    pip(*CORE, upgrade=FORCE_REINSTALL)
    if INSTALL_SEGMENTATION:
        # One command first, so pip resolves ONE consistent set of versions. If that
        # fails, retry package by package — a single unresolvable dependency must not
        # leave the whole stack uninstalled, and its error has to be visible.
        print("python: segmentation stack (~5 min)")
        if not pip(*SEGMENTATION, upgrade=FORCE_REINSTALL):
            print("  combined install failed — retrying one by one to find the culprit")
            for pkg in SEGMENTATION:
                pip(pkg, upgrade=FORCE_REINSTALL)

print("\nversions:")
still_missing = []
for mod in ["torch", "numpy"] + REQUIRED:
    ok, detail = importable(mod)
    print(f"  {mod:18s} {detail if ok else 'NOT IMPORTABLE — ' + detail}")
    if not ok and mod in REQUIRED:
        still_missing.append(mod)

if still_missing:
    print("\n" + "=" * 70)
    print(f"STOP: {', '.join(still_missing)} not importable after install.")
    print("  1) Runtime -> Restart session (once), then run cells 1 and 2 again.")
    print("  2) If still missing, the pip errors above name the reason — send them.")
    print("  Do NOT run cell 3 yet: every stage would skip and the report would be empty.")
    print("=" * 70)
else:
    print("\nвсе инструменты на месте — можно запускать ячейку 3")

try:
    import torch
    if torch.cuda.is_available():
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"\nGPU: {torch.cuda.get_device_name(0)} ({vram_gb:.0f} GB)")
        if vram_gb >= 24:
            print("  Хватает на профиль QUALITY в ячейке 3 (кросс-проверка второй моделью")
            print("  + зеркальная TTA на устойчивость определения стороны).")
        else:
            print("  Профиль QUALITY лучше не включать: он рассчитан на 24+ ГБ VRAM.")
    else:
        print("\nNO GPU — Runtime → Change runtime type → GPU. Сегментация на CPU займёт часы.")
except Exception:
    pass

In [ ]:
# @title 3 · Запуск пайплайна {display-mode:"form"}
DICOM_PATH = ""  # @param {type:"string"}
# Пусто = найти само: Drive (…/MyDrive/mri/*.zip), /content, файл study_source.txt
# в кэше, переменная SPINELAB_STUDY, либо единственный .zip в самом репозитории.
# Заполнять только чтобы указать конкретный архив.
SUBJECT_ID = "anon"  # @param {type:"string"}
STAGES = "all"  # @param ["all", "ingest", "ingest,spineps", "register,fatsat_qc,facets_axial,posterior,marrow,report", "geometry,muscles,canal,discs,radiomics,report", "report"]
FORCE = ""  # @param {type:"string"}
QUALITY = True  # @param {type:"boolean"}
TTA_MIRROR = True  # @param {type:"boolean"}
SHOW_PHI_AUDIT = True  # @param {type:"boolean"}

# QUALITY: тратит GPU-время на надёжность, а не на скорость — независимая вторая
# модель позвонков (TotalSegmentator vertebrae_mr) даёт согласие по уровням.
# TTA_MIRROR: повторный прогон на зеркальной копии проверяет, устойчиво ли модель
# определяет лево/право. Ни то, ни другое не меняет определения измерений —
# только то, насколько мы знаем их надёжность. Для 24+ ГБ VRAM.

import json, sys
from pathlib import Path

for p in ("/content/spinelab-src",):
    if p not in sys.path:
        sys.path.insert(0, p)

from spinelab.config import DEFAULT_STAGES, Config
from spinelab.pipeline import run_pipeline

# --- предполётная проверка: найти исследование и убедиться, что оно читаемо ----
from spinelab.discover import discover

found = discover(DICOM_PATH, cache_dir=CACHE_DIR)
if not found.source:
    raise SystemExit(
        f"СТОП: исследование не найдено ({found.how}).\n"
        "Любой из вариантов:\n"
        "  1) положить архив в /content/drive/MyDrive/mri/\n"
        f"  2) записать путь или ссылку в {CACHE_DIR}/study_source.txt\n"
        "  3) вписать путь или ссылку в поле DICOM_PATH выше\n"
        "Без данных все этапы будут пропущены, а прогон прерван.")
print(f"источник: {found.source}\n  найден: {found.how}")
DICOM_PATH = found.source

if DICOM_PATH.startswith(("http://", "https://")):
    import urllib.request
    try:
        with urllib.request.urlopen(
                urllib.request.Request(DICOM_PATH, method="HEAD"), timeout=30) as r:
            print(f"  ссылка доступна, {int(r.headers.get('Content-Length') or 0)/1e6:.1f} МБ")
    except Exception as exc:
        raise SystemExit(f"СТОП: ссылка недоступна — {exc}")
else:
    print(f"  {Path(DICOM_PATH).stat().st_size / 1e6:.1f} МБ")

stages = DEFAULT_STAGES if STAGES == "all" else tuple(s.strip() for s in STAGES.split(",") if s.strip())
config = Config(
    dicom_source=DICOM_PATH,
    subject_id=SUBJECT_ID,
    work_dir=Path("/content/spine_work"),
    cache_dir=Path(CACHE_DIR),
    stages=stages,
    force=tuple(s.strip() for s in FORCE.split(",") if s.strip()),
    quality=QUALITY,
    tta_mirror=TTA_MIRROR,
)
results = run_pipeline(config)

print("\n" + "=" * 62)
for name, res in results.items():
    print(f"{name:18s} {res.status.value:9s} {res.reason or ''}"[:120])

if SHOW_PHI_AUDIT:
    audit = json.loads((config.results_dir / "phi_audit.json").read_text(encoding="utf-8")) \
        if (config.results_dir / "phi_audit.json").exists() else None
    if audit:
        print("\nИдентифицирующие теги в DICOM:")
        print(" ", audit["verdict"])
        print("  теги:", ", ".join(audit["phi_tags_present"]) or "—")
        print("  деидентифицировать:  !python -m spinelab deid --dicom <dir> --out /content/anon")

In [ ]:
# @title 4 · Отчёт {display-mode:"form"}
COPY_TO_DRIVE = True  # @param {type:"boolean"}
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/spinelab-results"  # @param {type:"string"}

import shutil
from pathlib import Path
from IPython.display import HTML, display

report = Path("/content/spine_work/results/report.html")
if not report.exists():
    print("Отчёта нет — запустите ячейку 3 (этап report).")
else:
    if COPY_TO_DRIVE:
        dest = Path(DRIVE_RESULTS_DIR) / Path("/content/spine_work/results").name
        # Копируем только результаты (JSON/HTML/PNG), не промежуточные NIfTI:
        # они большие, а Drive-квота и канал ограничены.
        for src in Path("/content/spine_work/results").rglob("*"):
            if src.is_file() and src.suffix.lower() in (".html", ".json", ".png", ".csv"):
                out = dest / src.relative_to("/content/spine_work/results")
                out.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src, out)
        print(f"скопировано в {dest}")
    display(HTML(report.read_text(encoding="utf-8")))

In [ ]:
# @title 5 · Диагностика — это и надо присылать при проблемах {display-mode:"form"}
LOG_TAIL = 60  # @param {type:"integer"}
SHOW_KEY_NUMBERS = True  # @param {type:"boolean"}
COPY_LOGS_TO_DRIVE = True  # @param {type:"boolean"}

import shutil, subprocess, sys
from pathlib import Path

cmd = [sys.executable, "-m", "spinelab", "diagnose", "--work", "/content/spine_work",
       "--log-tail", str(LOG_TAIL)]
if SHOW_KEY_NUMBERS:
    cmd.append("--numbers")
print(subprocess.run(cmd, capture_output=True, text=True, cwd="/content/spinelab-src").stdout)

if COPY_LOGS_TO_DRIVE:
    dest = Path(DRIVE_RESULTS_DIR) / "logs"
    dest.mkdir(parents=True, exist_ok=True)
    for name in ("run.log", "run.jsonl", "summary.json", "findings.json"):
        src = Path("/content/spine_work/results") / name
        if src.exists():
            shutil.copy2(src, dest / name)
            print(f"  {name} -> {dest / name}  ({src.stat().st_size/1024:.0f} КБ)")

### Быстрая проверка данных до запуска (10 секунд, без GPU)

```
!python -m spinelab inspect --dicom /content/drive/MyDrive/mri/study.zip
```

Печатает список серий (контраст, плоскость, срезы, TE/TR/TI) и вывод о том, на
какие вопросы это исследование может ответить, а на какие нет. Полезно посмотреть
до того, как тратить GPU.

### Порядок чтения отчёта

Сначала проверки, которые могут обесценить всё остальное: контуры сегментации →
подавление жира → устойчивость определения стороны → согласие двух моделей по
уровням. И только потом измерения. Подробно — `docs/COLAB.md`.

### Если что-то пошло не так

| Симптом | Что делать |
|---|---|
| Colab отключился на середине | Запустить ячейку 3 снова: выполненные этапы читаются из `results/stages/*.json`, продолжится с места обрыва. |
| Нужно пересчитать один этап | В поле `FORCE` через запятую: `marrow,posterior`. |
| `spineps` не импортируется | Ячейка 2 с `FORCE_REINSTALL`, затем Runtime → Restart session (вручную, один раз). |
| Веса качаются каждый раз | Проверьте, что `CACHE_DIR` в Drive и Drive смонтирован; ячейка 1 печатает размер кэша. |
| Этап `marrow` пропущен | В исследовании нет последовательности с подавлением жира — это ограничение данных, а не ошибка. См. `docs/clinical-context.md`. |
| Сравнение сторон «not_comparable» | Последовательность не покрывает обе стороны одинаково; разница была бы артефактом FOV. |
| `register` пропущен | Нет SimpleITK (ячейка 2) либо fat-sat серия и есть та же, что задаёт систему координат масок. Без него маски стоят по геометрии DICOM — работает, но без поправки на движение между сериями. |
| `register` не применил поправку | Оптимизатор дал неправдоподобный сдвиг (>15 мм или >10°) или не улучшил метрику — это признак неудачной регистрации, а не движения пациента. Оставляется выравнивание по заголовкам, и отчёт это пишет. |
| `crosscheck` пропущен | Профиль QUALITY выключен. |

Тесты (без GPU, ~1 с): `!python -m pytest /content/spinelab-src/tests -q`